In [66]:
import math
from datetime import timedelta
from pathlib import Path

import polars as pl
import polars_distance as pld
import polars_h3 as plh3
import yaml
from haversine import haversine

# https://ion-elgreco.github.io/polars-distance/api/dist/#polars_distance.DistancePairWise.haversine

with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [62]:
def rg(data: pl.DataFrame, result_column: str = "rg_time") -> pl.DataFrame:
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean")
        .alias("stop")
        .struct.rename_fields(["longitude", "latitude"]),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
        .alias("centroid")
        .struct.rename_fields(["longitude", "latitude"]),
    ).with_columns(
        pld.col("centroid").dist.haversine("stop", "km").alias("haversine"),
        pl.col.interval_lenght.dt.total_seconds().sum().alias("N"),
    )
    res = (
        ((d["haversine"] ** 2 * d["interval_lenght"].dt.total_seconds()).sum() / d["N"])
        .sqrt()
        .round(3)[0]
    )
    return pl.DataFrame({"device_id": [data["device_id"][0]], result_column: [res]})


def rg_place(data: pl.DataFrame, result_column: str = "rg_freq") -> pl.DataFrame:
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean")
        .alias("stop")
        .struct.rename_fields(["longitude", "latitude"]),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
        .alias("centroid")
        .struct.rename_fields(["longitude", "latitude"]),
    ).with_columns(
        pld.col("centroid").dist.haversine("stop", "km").alias("haversine"),
    )
    place_counts = d.group_by("place_label").len()
    N = place_counts["len"].sum()
    d = d.join(place_counts, on="place_label", how="inner").unique(
        subset=("place_label")
    )
    res = round(math.sqrt((d["haversine"] ** 2 * d["len"]).sum() / N), 3)
    return pl.DataFrame({"device_id": [data["device_id"][0]], result_column: [res]})


def rg_plh3(data: pl.DataFrame, result_column: str = "rg_time") -> pl.DataFrame:
    d = data.with_columns(
        pl.col.latitude_mean.mean().alias("c_lat"),
        pl.col.longitude_mean.mean().alias("c_lon"),
    ).with_columns(
        plh3.great_circle_distance(
            "latitude_mean",
            "longitude_mean",
            "c_lat",
            "c_lon",
            unit="km",
        ).alias("haversine"),
    )
    N = d["interval_lenght"].dt.total_seconds().sum()
    res = round(
        math.sqrt(
            (d["haversine"] ** 2 * d["interval_lenght"].dt.total_seconds()).sum() / N
        ),
        3,
    )
    return pl.DataFrame({"device_id": [data["device_id"][0]], result_column: [res]})


def rg_place_plh3(data: pl.DataFrame, result_column: str = "rg_freq") -> pl.DataFrame:
    d = data.with_columns(
        pl.col.latitude_mean.mean().alias("c_lat"),
        pl.col.longitude_mean.mean().alias("c_lon"),
    ).with_columns(
        plh3.great_circle_distance(
            "latitude_mean",
            "longitude_mean",
            "c_lat",
            "c_lon",
            unit="km",
        ).alias("haversine"),
    )
    place_counts = d.group_by("place_label").len()
    N = place_counts["len"].sum()
    d = d.join(place_counts, on="place_label", how="inner").unique(
        subset=("place_label")
    )
    res = round(math.sqrt((d["haversine"] ** 2 * d["len"]).sum() / N), 3)
    return pl.DataFrame({"device_id": [data["device_id"][0]], result_column: [res]})


def rg_hexagon(
    data: pl.DataFrame, res: int, result_column: str = "rg_hex"
) -> pl.DataFrame:
    c_lat = data["latitude_mean"].mean()
    c_lon = data["longitude_mean"].mean()
    d = data.with_columns(
        plh3.latlng_to_cell(c_lat, c_lon, resolution=res).alias("center"),
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), res)
        ).alias(f"h3_{res}"),
    ).with_columns(
        plh3.grid_distance("center", f"h3_{res}").alias("distance"),
        pl.col(f"h3_{res}").count().alias("N"),
    )
    place_counts = d.group_by(f"h3_{res}").len()
    N = place_counts["len"].sum()
    d = d.join(place_counts, on=f"h3_{res}", how="inner").unique(subset=(f"h3_{res}"))
    res = round(math.sqrt((d["distance"] ** 2 * d["len"]).sum() / N), 3)
    return pl.DataFrame({"device_id": [data["device_id"][0]], result_column: [res]})


In [4]:
DIR = f"{config['dirs']['output']}/{config['city']}"

In [7]:
test = pl.DataFrame(
    {
        "device_id": [1, 1, 1, 1, 1],
        "place_label": [1, 1, 2, 2, 2],
        "interval_lenght": [
            timedelta(minutes=20),
            timedelta(minutes=30),
            timedelta(minutes=25),
            timedelta(minutes=15),
            timedelta(minutes=10),
        ],
        "latitude_mean": [
            47.6612,
            47.6612,
            47.4984,
            47.4984,
            47.4984,
        ],
        "longitude_mean": [19.0764, 19.0764, 19.0404, 19.0404, 19.0404],
    }
)

In [8]:
c_lat = test["latitude_mean"].mean()
c_lon = test["longitude_mean"].mean()
p1 = haversine((c_lat, c_lon), (47.6612, 19.0764))
p2 = haversine((c_lat, c_lon), (47.4984, 19.0404))

In [9]:
round(
    math.sqrt(
        sum([i**2 * n for i, n in zip([p1, p1, p2, p2, p2], [20, 30, 25, 15, 10])])
        / 100
    ),
    3,
)

9.333

In [10]:
round(math.sqrt(sum([i**2 * n for i, n in zip([p1, p2], [2, 3])]) / 5), 3)

8.967

In [12]:
test.group_by("device_id").map_groups(rg)
test.group_by("device_id").map_groups(rg_place)

device_id,result
i64,f64
1,8.967


In [63]:
test.group_by("device_id").map_groups(rg_place_plh3)

device_id,rg_freq
i64,f64
1,8.969


In [13]:
for _ in range(1000):
    test.group_by("device_id").map_groups(rg)

In [57]:
rg_plh3(test)

6000


device_id,rg_time
i64,f64
1,9.336


In [59]:
for _ in range(1000):
    test.group_by("device_id").map_groups(rg_plh3)

In [16]:
rg_plh3(test)

device_id,place_label,interval_lenght,latitude_mean,longitude_mean,result
i64,i64,duration[μs],f64,f64,f64
1,1,20m,47.6612,19.0764,9.336


In [29]:
rg_hexagon(test, 8)

device_id,result
i64,f64
1,11.764


run for all

In [ ]:
Path(f"{DIR}/descriptives/radius_of_gyration/").mkdir(exist_ok=True, parents=True)
# acc = pl.DataFrame()
for i in range(128, 256):
    temp = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")
    rg_time = temp.group_by("device_id").map_groups(rg_plh3)
    rg_freq = temp.group_by("device_id").map_groups(rg_place_plh3)
    rg_time.join(rg_freq, on="device_id").sort("device_id").write_csv(
        f"{DIR}/descriptives/radius_of_gyration/{i:02x}.csv"
    )
    # acc = pl.concat([acc, rg_time.join(rg_freq, on="device_id")])
# acc = acc.sort("device_id")

# 16 partitions run about 7m
# 96 partitions run about 42m
# 128 partitions run about 58m

In [72]:
acc = pl.DataFrame()
for i in range(256):
    g = pl.read_csv(f"{DIR}/descriptives/radius_of_gyration/{i:02x}.csv")
    acc = pl.concat([acc, g])
acc = acc.sort("device_id")
acc.write_csv(f"{DIR}/descriptives/radius_of_gyration/full.csv")